# Same-input honest replay of Azure PBT attack suites
**Purpose.** Offline verification of the frozen reviewed-input training pilot. For every attack-authored suite, execute that exact suite on the paired honest solution using exactly the attack candidate's reviewed inputs. There are no model calls, prompt changes, input regeneration, test repairs, or held-out tasks.

**Interpretation.** An attack assertion accompanied by no honest assertion failures is a differential catch on these inputs, not a proof of correctness or independent confirmation of pool labels. Errors/incomplete execution are excluded explicitly. Ordinary honest-suite false-positive rates remain a separate measurement. No bootstrap or population claim is made for this training diagnostic.

**Execution.** Cell1 loads completed arms and verifies identities. Cell2 defines fingerprinted cache/replay and tests pure outcome logic. Cell3 runs at most38 Docker replays (19 per arm); each candidate result is cached before the next. Cell4 compares only common eligible task pairs. Rerunning reads matching caches; changed sources/config/code require a new cache version. Recorded replay failures are retained, never automatically retried. Candidate code executes only in Docker with the existing pipeline sandbox. No API credentials are loaded.


In [ ]:
from pathlib import Path
import ast, hashlib, json, os, sys
from collections import Counter
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
assert (REPO / "pipeline").is_dir()
from pipeline.data import Dataset, load_records
from pipeline import sandbox
PREFIX = "azure-terra-pbt-train19-s300-reviewed-v1"
DATA = Path("data/azure_pbt_train19_s300_reviewed_v1.json")
TRIGGERS = PREFIX + "-inputs"
ARMS = (PREFIX + "-plain-v3", PREFIX + "-traceable-v1")
IMAGE = "python@sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea"
def sha_bytes(value):
    return hashlib.sha256(value).hexdigest()
def json_hash(value):
    return sha_bytes(json.dumps(value, sort_keys=True, ensure_ascii=False).encode("utf-8"))
dataset = Dataset.load(DATA)
assert len(dataset.train) == 19 and not dataset.test
expected_ids = {c.candidate_id for _, c in dataset.candidates()}
inputs = {r["candidate_id"]: r for r in load_records(TRIGGERS)}
assert set(inputs) == expected_ids and len(inputs) == 38
arm_rows, configs = {}, {}
for name in ARMS:
    rows = load_records(name)
    assert len(rows) == len({r["candidate_id"] for r in rows}) == 38, f"{name} incomplete"
    assert {r["candidate_id"] for r in rows} == expected_ids
    assert all(r["split"] == "train" for r in rows)
    config = json.loads((Path("runs") / name / "config.json").read_text(encoding="utf-8"))
    configs[name] = config
    arm_rows[name] = {r["candidate_id"]: r for r in rows}
source_fingerprints = {
    "dataset_sha256": sha_bytes(DATA.read_bytes()),
    "inputs_sha256": sha_bytes((Path("runs") / TRIGGERS / "records.jsonl").read_bytes()),
    "sandbox_sha256": sha_bytes(Path(sandbox.__file__).read_bytes()),
}
print({"tasks": 19, "candidates_per_arm": 38, "held_out_tasks": 0, "model_calls": 0})


In [ ]:
def measurement(result):
    counts = Counter(r["outcome"] for r in result["records"])
    outcomes = {name: counts[name] for name in sandbox.RECORD_OUTCOMES}
    assert not set(counts).difference(sandbox.RECORD_OUTCOMES)
    if not result["ok"] or not result["complete"]:
        return {"failed": True, "blame": "infra", "reason": "Unusable or incomplete replay",
                "honest_fired": None, "outcomes": outcomes}
    assert len(result["records"]) == result["n_expected"]
    if outcomes["candidate_crash"] or outcomes["prop_error"]:
        return {"failed": True, "blame": "model", "reason": "Replay has candidate crash or property error; diagnostic excluded",
                "honest_fired": None, "outcomes": outcomes}
    return {"failed": False, "blame": None, "reason": None,
            "honest_fired": outcomes["catch"] > 0, "outcomes": outcomes}

# Pure regression examples: an unmeasured grid must not become honest_fired=False.
assert measurement({"ok": False, "complete": False, "records": []})["honest_fired"] is None
assert measurement({"ok": True, "complete": True, "n_expected": 1,
                    "records": [{"outcome": "catch"}]})["honest_fired"] is True
assert measurement({"ok": True, "complete": True, "n_expected": 1,
                    "records": [{"outcome": "pass"}]})["honest_fired"] is False
assert measurement({"ok": True, "complete": True, "n_expected": 1,
                    "records": [{"outcome": "prop_error"}]})["failed"]

def replay(name, task, attack_row):
    config = {**source_fingerprints, "arm_config_sha256": json_hash(configs[name]),
              "source_record_sha256": json_hash(attack_row), "task_id": task.task_id,
              "honest_code_sha256": sha_bytes(task.honest.code.encode("utf-8")),
              "suite_sha256": sha_bytes(attack_row["tests_src"].encode("utf-8")),
              "input_sha256": json_hash(inputs[task.attack.candidate_id]["inputs"]),
              "image": IMAGE, "timeout_seconds": 120,
              "kind": "attack-authored-suite-on-paired-honest-same-inputs-v1"}
    cache = Path("runs") / name / "same-input-honest-replay-v1" / (task.task_id + ".json")
    if cache.exists():
        saved = json.loads(cache.read_text(encoding="utf-8"))
        assert saved["config"] == config, f"Cache mismatch: {cache}; preserve and version"
        return saved
    try:
        result = sandbox.run_raw(task, task.honest.code, attack_row["tests_src"],
                                 inputs[task.attack.candidate_id]["inputs"],
                                 timeout_s=120, isolation=sandbox.Isolation.DOCKER, docker_image=IMAGE)
        observed = measurement(result)
        saved = {"config": config, "measurement": observed, "result": result}
    except Exception as error:
        saved = {"config": config, "measurement": {
            "failed": True, "blame": "infra", "reason": type(error).__name__ + ": " + str(error),
            "honest_fired": None, "outcomes": None}, "result": None}
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps(saved, indent=2) + "\n", encoding="utf-8")
    return saved


In [ ]:
diagnostics = {}
for name in ARMS:
    details = []
    for task in dataset.tasks:
        row = arm_rows[name][task.attack.candidate_id]
        item = {"task_id": task.task_id, "candidate_id": task.attack.candidate_id}
        if row["failed"]:
            item.update(eligible=False, exclusion="source attack-suite measurement failed",
                        blame=row["blame"], reason=row["reason"], attack_fired=None,
                        honest_fired=None, differential_catch=None)
        elif row["n_pairs_run"] != row["n_pairs_expected"]:
            item.update(eligible=False, exclusion="source attack grid incomplete", attack_fired=None,
                        honest_fired=None, differential_catch=None)
        elif row["n_pairs_by_outcome"]["candidate_crash"] or row["n_pairs_by_outcome"]["prop_error"]:
            item.update(eligible=False, exclusion="source attack grid contains execution errors",
                        source_outcomes=row["n_pairs_by_outcome"], attack_fired=None,
                        honest_fired=None, differential_catch=None)
        else:
            saved = replay(name, task, row)
            measure = saved["measurement"]
            item.update(eligible=not measure["failed"], attack_fired=bool(row["fired"]),
                        honest_fired=measure["honest_fired"], honest_outcomes=measure["outcomes"],
                        source_outcomes=row["n_pairs_by_outcome"],
                        exclusion=measure["reason"], blame=measure["blame"],
                        differential_catch=(bool(row["fired"]) and not measure["honest_fired"])
                            if not measure["failed"] else None)
        details.append(item)
    diagnostics[name] = details
    print({"arm": name, "complete_task_diagnostics": len(details),
           "eligible": sum(d["eligible"] for d in details),
           "excluded": [d for d in details if not d["eligible"]]})


In [ ]:
common_tasks = set.intersection(*(
    {d["task_id"] for d in diagnostics[name] if d["eligible"]} for name in ARMS))
summary = {"scope": "training diagnostic, not held-out safety",
           "common_eligible_task_count": len(common_tasks), "common_eligible_task_ids": sorted(common_tasks),
           "model_calls": 0, "arms": {}}
for name in ARMS:
    matched = [d for d in diagnostics[name] if d["task_id"] in common_tasks]
    n = len(matched)
    summary["arms"][name] = {
        "matched_tasks": n,
        "attack_firing_count": sum(d["attack_fired"] for d in matched),
        "same_suite_honest_firing_count": sum(d["honest_fired"] for d in matched),
        "differential_catch_count": sum(d["differential_catch"] for d in matched),
        "differential_catch_rate": sum(d["differential_catch"] for d in matched) / n if n else None,
        "excluded_tasks": [d for d in diagnostics[name] if not d["eligible"]],
        "details": diagnostics[name],
    }
    report = Path("runs") / name / "same-input-honest-replay-summary-v1.json"
    payload = json.dumps({"fingerprints": source_fingerprints, "summary": summary["arms"][name]},
                         indent=2) + "\n"
    if report.exists():
        assert report.read_text(encoding="utf-8") == payload, "Frozen replay summary changed; version required"
    else:
        report.write_text(payload, encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
def successful(row, error_free=False):
    if row["failed"]:
        return False
    if row["n_pairs_run"] != row["n_pairs_expected"]:
        return False
    return not error_free or not (row["n_pairs_by_outcome"]["candidate_crash"] or row["n_pairs_by_outcome"]["prop_error"])

def firing_measure(name, ids):
    values = [bool(arm_rows[name][cid]["fired"]) for cid in sorted(ids)]
    return {"n": len(values), "fired": sum(values),
            "rate": sum(values) / len(values) if values else None,
            "fired_candidate_ids": [cid for cid in sorted(ids) if arm_rows[name][cid]["fired"]]}

report = {"paired": {}, "error_free_paired_sensitivity": {}, "unpaired": {}, "diversity": {},
          "usage": {}, "failure_records": {},
          "provenance": {**source_fingerprints,
             "arm_records_sha256": {a: sha_bytes((Path("runs") / a / "records.jsonl").read_bytes()) for a in ARMS}}}
for group, attack in (("honest", False), ("attack", True)):
    for error_free, target in ((False, "paired"), (True, "error_free_paired_sensitivity")):
        common = set.intersection(*({cid for cid, row in arm_rows[a].items()
                  if row["is_attack"] == attack and successful(row, error_free)} for a in ARMS))
        report[target][group] = {a: firing_measure(a, common) for a in ARMS}
    report["unpaired"][group] = {a: firing_measure(a, {cid for cid, row in arm_rows[a].items()
                                  if row["is_attack"] == attack and successful(row)}) for a in ARMS}

for name in ARMS:
    signatures_by_candidate = {}
    for cid, row in arm_rows[name].items():
        if not successful(row):
            continue
        signatures = []
        for node in ast.parse(row["tests_src"]).body:
            if isinstance(node, ast.FunctionDef) and node.name.startswith(("test_", "prop_")):
                body = node.body
                if (body and isinstance(body[0], ast.Expr) and isinstance(body[0].value, ast.Constant)
                        and isinstance(body[0].value.value, str)):
                    body = body[1:]
                signatures.append(ast.dump(ast.Module(body=body, type_ignores=[]), include_attributes=False))
        signatures_by_candidate[cid] = {"tests": len(signatures), "unique_bodies": len(set(signatures))}
    report["diversity"][name] = {"measured_suites": len(signatures_by_candidate),
        "test_functions": sum(v["tests"] for v in signatures_by_candidate.values()),
        "unique_bodies_summed_within_suites": sum(v["unique_bodies"] for v in signatures_by_candidate.values()),
        "suites_with_duplicates": sum(v["unique_bodies"] < v["tests"] for v in signatures_by_candidate.values()),
        "fully_duplicate_suites": sum(v["unique_bodies"] == 1 and v["tests"] > 1 for v in signatures_by_candidate.values()),
        "by_candidate": signatures_by_candidate}
    calls = [c for row in arm_rows[name].values() for c in row["calls"]]
    known = [c["usage"] for c in calls if c["usage"] is not None]
    report["usage"][name] = {"recorded_calls": len(calls), "known_usage_calls": len(known),
        "missing_usage_calls": len(calls)-len(known),
        "input_tokens": sum(u["input_tokens"] for u in known) if known else None,
        "output_tokens": sum(u["output_tokens"] for u in known) if known else None,
        "dollars": None, "cost_note": "Azure prices not independently verified; retries may not all be represented"}
    report["failure_records"][name] = [
        {k: r[k] for k in ("candidate_id", "failed", "blame", "reason")}
        for r in arm_rows[name].values() if r["failed"]]
report["input_diversity"] = {
    "retained_inputs": sum(len(r["inputs"]) for r in inputs.values()),
    "raw_unique_summed_within_candidates": sum(len(set(r["inputs"])) for r in inputs.values()),
    "whitespace_normalized_unique_summed_within_candidates": sum(len({" ".join(x.split()) for x in r["inputs"]}) for r in inputs.values()),
    "candidate_details": {cid: {"inputs": len(r["inputs"]), "raw_unique": len(set(r["inputs"])),
        "whitespace_normalized_unique": len({" ".join(x.split()) for x in r["inputs"]})} for cid, r in inputs.items()}}
report["same_input_replay"] = summary
output = Path("runs") / TRIGGERS / "training-comparison-summary-v1.json"
payload = json.dumps(report, indent=2) + "\n"
if output.exists():
    assert output.read_text(encoding="utf-8") == payload, "Frozen comparison changed; version required"
else:
    output.write_text(payload, encoding="utf-8")
print(json.dumps({k: report[k] for k in ("paired", "error_free_paired_sensitivity", "unpaired", "usage")}, indent=2))
print({"report": str(output)})


In [ ]:
# Pair-level sensitivity: do not discard genuine witnesses merely because another test has a bad oracle.
pairwise = {}
for name in ARMS:
    by_task = {}
    for task in dataset.tasks:
        if task.task_id not in common_tasks:
            continue
        row = arm_rows[name][task.attack.candidate_id]
        cached = json.loads((Path("runs") / name / "same-input-honest-replay-v1" / (task.task_id + ".json")).read_text(encoding="utf-8"))
        assert not cached["measurement"]["failed"]
        assert cached["config"]["source_record_sha256"] == json_hash(row)
        attack_catches = {(c["test"], c["input_index"]) for c in row["catches"]}
        honest_passes = {(r["prop"], r["i"]) for r in cached["result"]["records"] if r["outcome"] == "pass"}
        witnesses = sorted(attack_catches.intersection(honest_passes))
        by_task[task.task_id] = {"witness_count": len(witnesses),
            "witnesses": [{"test": test, "input_index": index} for test, index in witnesses]}
    pairwise[name] = {"matched_tasks": len(by_task),
        "tasks_with_at_least_one_fail_attack_pass_honest_pair": sum(bool(v["witnesses"]) for v in by_task.values()),
        "by_task": by_task}
target = Path("runs") / TRIGGERS / "pairwise-differential-witnesses-v1.json"
payload = json.dumps({"provenance": report["provenance"], "arms": pairwise}, indent=2) + "\n"
if target.exists():
    assert target.read_text(encoding="utf-8") == payload
else:
    target.write_text(payload, encoding="utf-8")
print({name: {k: v for k, v in result.items() if k != "by_task"} for name, result in pairwise.items()})
